In [2]:
import numpy as np
from sklearn.neighbors import kneighbors_graph # kneighbors_graph takes a dataset and builds a graph where each point is connected only to its k nearest neighbors. it returns a sparse matrix for efficiency issues
from scipy.sparse.csgraph import shortest_path # csgraph stands for compressed sparse graph. this fn takes a sparse graph - the knn graph and computes the shortest path between every pair of nodes - which is exactly what turns local euclidean edge weights into global geodesic distances
from scipy.sparse import csr_matrix # imports the csr_matrix => csr stands for compressed sparse row... a specific memory format for sparse matrices where row indices, column indices and values are stored in compact arrays. shortest_path requires its input to be in this format so we need this as a converter

data setup:

In [3]:
# X : shape (M,N) -> M samples, N features
X = np.random.randn(100,10) # 100 points in 10-D space | randn draws from a gaussian with mean of zero and standard dev of 1
M = X.shape[0] 

build the k-nearest neighbor graph:
- this fn computes the pairwise euclidean distances across all points in the dataset to figure out who is near to whom. 
- n_neighbors=k | how many neighbors each point connects to. if k=10, each point gets exactly 10 outgoing edges. This controls the density of the graph.. too small and the graph becomes disconnected (islands with no path between them, producing inf in the distance matrix) | too large and distances short circuit
- mode='distance' | this determines what values get stored on each edge | distance stores the actual euclidean distance between two points | the alternative, connectivity just stores a 1 or 0 (connected or not)
- include_self = False -> whether to include each point as its own neighbor. 
- the returned matrix is of shape MxM -> where each position in the matrix is and holds the euclidean distance between two points if they're neighbors and zero otherwise. 
- so kneighbors_graph returns an MxM graph where the positions within the graph > 0 signify an edge exisiting between the points and that numeric figure is the euclidean distance between the points... 
- this is what the kneighbors_graph algo does under the hood for better explanation:
- "compute the normal euclidean distance between all points...producing a dense MxM matrix...then filter it in such a way that for each row..retain the smallest k euclidean distances whilst setting all other values to zero..."

In [15]:
k = 10
A = kneighbors_graph(X, n_neighbors=k, mode='distance', include_self=False)

In [18]:
# need to symmetrize the matrix...much as the euclidean distances are symmetric...the selection of the k-nearest neighbors isn't and this is how we set and ensure symmetry is maintained
A = A.maximum(A.T)

run the shortest paths algorithm:
- csr_matrix(A) => converts A into CSR (Compressed Sparse Row format) which shortest_path requires. even if A is already sparse, this ensures its in the format expected
- method = 'fw' => USES THE FLOYD-WARSHALL ALGORITHM..(SIMPLE AND EXACT BUT SLOW FOR large M) => use when M is small (under 1000) | could also use method='D' - selects Djikstra...
- directed = False | treats the graph as undirected...
- output D is a dense MxM numpy array....where D{i,j} is the length of the shortest path from point i to j along the graph edges..... this is the geodesic approximation

In [21]:
D = shortest_path(csr_matrix(A), method='D', directed=False)

In [22]:
D

array([[0.        , 5.92326099, 3.13622323, ..., 6.71956151, 5.66575419,
        6.37737151],
       [5.92326099, 0.        , 5.38267264, ..., 3.34975423, 3.46294201,
        6.99967995],
       [3.13622323, 5.38267264, 0.        , ..., 5.40689033, 6.10914143,
        5.55390501],
       ...,
       [6.71956151, 3.34975423, 5.40689033, ..., 0.        , 6.81269624,
        3.64992572],
       [5.66575419, 3.46294201, 6.10914143, ..., 6.81269624, 0.        ,
        8.7930635 ],
       [6.37737151, 6.99967995, 5.55390501, ..., 3.64992572, 8.7930635 ,
        0.        ]], shape=(100, 100))

In [23]:
print(D.shape)

(100, 100)


In [26]:
print(D[50,50]) # distance to self is zero obviously...

0.0


In [27]:
print(D[0,1]) # geodesic distance between point 0 and 1

5.923260992163486


In [ ]:
assert np.allclose(D, D.T) # symmetric | numpy.allclose checks whether 2 arrays are approximately equal element by element..not exactly equal but close enough equal

In [30]:
assert np.all(np.diag(D) == 0) # zero diagonal

In [31]:
assert np.all(D >= 0)